# 02 Sweep design: frozen grids and live smoke test

The three committed sweep grids (13,290 configs at N=1000) are defined once in `src/run_sweeps.py` and executed by that script, not here: full grids take ~30-45 min on 12 workers and `run_sweep()` resumes across invocations. This notebook documents the design, checks the config counts, and runs one live configuration per grid as a smoke test.

| Grid | Topologies | Rules | α values | δ values | β | Seeds | Configs |
|---|---|---|---|---|---|---|---|
| `pilot_full` | ws, ba, er | observation, imitation, broadcast, broadcast_hub, none | 0, 0.25, 0.5, 0.75, 1 | 0, 0.05, 0.15, 0.3, 0.6 | 0.12 | 30 | 9,090 |
| `zoom_h2` | ws, ba, er | broadcast, broadcast_hub | 0.02, 0.05, 0.08, 0.1, 0.15 | 0, 0.05, 0.15, 0.3 | 0.12 | 30 | 3,600 |
| `beta_sweep` | ws, ba, er | none, observation | 0.75 | 0.1 | 0.03-0.2 (5) | 20 | 600 |

Run the full grids from the repo root: `python -m src.run_sweeps --all --workers 12 --out-dir data`. `none` keeps a single (α, δ) cell per topology; broadcast rules map α to compliance probability `p_comply`.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.run_sweeps import pilot_configs, zoom_configs, beta_configs

counts = {"pilot_full": len(pilot_configs()), "zoom_h2": len(zoom_configs()),
          "beta_sweep": len(beta_configs())}
print(counts)
assert counts == {"pilot_full": 9090, "zoom_h2": 3600, "beta_sweep": 600}
print("grid sizes reproduce the committed row counts")

{'pilot_full': 9090, 'zoom_h2': 3600, 'beta_sweep': 600}
grid sizes reproduce the committed row counts


In [2]:
import numpy as np
from src.simulation import SimParams, run_coupled

# one live configuration per grid (same seeds/params as the frozen grids)
smoke = [("pilot_full", dict(topology="ws", rule="observation", alpha=1.0, delta=0.0, seed=0)),
          ("zoom_h2", dict(topology="ba", rule="broadcast_hub", alpha=0.1, delta=0.0, seed=3)),
          ("beta_sweep", dict(topology="ws", rule="none", alpha=0.75, delta=0.1, seed=0, beta=0.03))]
for name, kw in smoke:
    out = run_coupled(SimParams(n=1000, **kw))
    assert np.isfinite(out["final_size"]) and 0.0 <= out["final_size"] <= 1.0
    print(f"{name}: {kw['rule']} final={out['final_size']:.3f} waves={out['waves']}")
print("smoke subset runs finite: pipeline is live")

pilot_full: observation final=0.067 waves=1


zoom_h2: broadcast_hub final=0.891 waves=1


beta_sweep: none final=0.757 waves=2
smoke subset runs finite: pipeline is live
